# Predicting Bach Chorales

In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

### Data Extraction

In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.layers import Reshape
import pandas as pd
import pickle
import numpy as np
import os
import sys
from sklearn.model_selection import train_test_split

2025-03-18 15:10:17.756223: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742325017.786677  108562 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742325017.800869  108562 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
data_root = 'data/'
drive_root = '/content/drive/My Drive/Colab Notebooks/Bach Chorales/'
window_size = 48

In [4]:
def get_data(first=False):
    df = pd.DataFrame()
    if not first:
        df = pickle.load(open(data_root + "bach.pkl", "rb"))
        return df
    folders = ["test", "train", "valid"]
    for folder in folders:
        for file in os.listdir(data_root + "jsb_chorales/" + folder):
            if file.endswith(".csv"):
                df = pd.concat([df, pd.read_csv(data_root + "jsb_chorales/" + folder + "/" + file)])
    pickle.dump(df, open(data_root + "bach.pkl", "wb"))
    return df

def get_Xy(df):
    X = []
    y = []
    for i in range(len(df) - window_size):
        X.append(df.iloc[i:i+window_size].values)
        y.append(df.iloc[i+window_size].values)
    return np.array(X), np.array(y)

In [5]:
df = get_data(True)
X, y = get_Xy(df)

In [6]:
X.shape, y.shape

((92488, 48, 4), (92488, 4))

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((73990, 48, 4), (73990, 4), (18498, 48, 4), (18498, 4))

In [22]:
np.max(y_test)

np.int64(81)

In [ ]:
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], X_train.shape[2])
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], X_test.shape[2])

y_train = keras.utils.to_categorical(y_train, num_classes=128)
y_test = keras.utils.to_categorical(y_test, num_classes=128)

In [9]:
X_train[0]

array([[69, 66, 61, 54],
       [71, 68, 62, 47],
       [71, 68, 62, 47],
       [71, 68, 62, 47],
       [71, 68, 62, 47],
       [68, 65, 61, 49],
       [68, 65, 61, 49],
       [68, 65, 61, 49],
       [68, 65, 61, 49],
       [68, 65, 61, 49],
       [68, 65, 61, 49],
       [68, 65, 61, 49],
       [68, 65, 61, 49],
       [69, 66, 61, 54],
       [69, 66, 61, 54],
       [69, 66, 61, 54],
       [69, 66, 61, 54],
       [71, 68, 64, 52],
       [71, 68, 64, 52],
       [71, 68, 64, 52],
       [71, 68, 64, 52],
       [73, 69, 64, 57],
       [73, 69, 64, 57],
       [73, 69, 64, 57],
       [73, 69, 64, 57],
       [71, 64, 64, 56],
       [71, 64, 64, 56],
       [73, 64, 64, 56],
       [73, 64, 64, 56],
       [74, 69, 62, 54],
       [74, 69, 62, 54],
       [74, 69, 62, 54],
       [74, 69, 62, 54],
       [73, 69, 64, 52],
       [73, 69, 64, 52],
       [73, 69, 64, 52],
       [73, 69, 64, 52],
       [71, 69, 66, 50],
       [71, 69, 66, 50],
       [71, 69, 66, 50],


In [10]:
model = Sequential([
    Input(shape=(window_size, 4)),
    GRU(256, return_sequences=True),
    GRU(256, return_sequences=False),
    Dense(4),
])

I0000 00:00:1742325065.518438  108562 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2256 MB memory:  -> device: 0, name: NVIDIA T500, pci bus id: 0000:01:00.0, compute capability: 7.5


In [11]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 48, 256)        │       201,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 256)            │       394,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 596,996 (2.28 MB)

 Trainable params: 596,996 (2.28 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
from tensorflow.keras.callbacks import ModelCheckpoint
checkpoint_filepath = data_root + 'best_bach_model.keras'
#'/content/drive/My Drive/Colab Notebooks/Bach Chorales/best_bach_model.keras'

model_checkpoint_callback = ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=False,  # Save the entire model
    monitor='val_loss',  # Monitor validation loss
    mode='min',  # Save the model when val_loss is minimized
    save_best_only=True  # Only save the best model
)

model.compile(optimizer='adam', loss='mse', metrics=['accuracy'])
history = model.fit(X_train, y_train, epochs=2, batch_size=128, validation_split=0.2, callbacks=[model_checkpoint_callback])

Epoch 1/2
463/463 ━━━━━━━━━━━━━━━━━━━━ 71s 144ms/step - accuracy: 0.0036 - loss: 11.4570 - val_accuracy: 0.0034 - val_loss: 7.3924
Epoch 2/2
463/463 ━━━━━━━━━━━━━━━━━━━━ 77s 165ms/step - accuracy: 0.0035 - loss: 6.9840 - val_accuracy: 0.0034 - val_loss: 6.2855


In [15]:
def next_chord(chord):
    nc_unrounded = model.predict(chord.reshape(1, window_size, 4), verbose=0)
    nc = np.round(nc_unrounded).reshape(1, 4)
    return nc

def extend_sequence(sequence, length):
    for i in range(length):
        next_chord_result = next_chord(sequence[-window_size:])
        sequence = np.concatenate((sequence, next_chord_result), axis=0)
    return sequence

In [17]:
%pip install mido

  Using cached mido-1.3.3-py3-none-any.whl.metadata (6.4 kB)
Using cached mido-1.3.3-py3-none-any.whl (54 kB)

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
import os
import csv
import numpy as np
from mido import Message, MidiFile, MidiTrack
import argparse
import tempfile
import shutil
import tarfile
import requests
from io import BytesIO

def quartet_to_midi(chorale, midi_file, tempo=120):
		"""Convert an SATB array from the JSB Chorales dataset to MIDI."""
		# Create a new MIDI file with a single track
		mid = MidiFile()


		# Set tempo (500000 microseconds per beat = 120 BPM)
		tempo_value = 500000  # in microseconds per beat

		# Each row in the array represents a time step (e.g., a 16th note)
		# Each column represents a voice (soprano, alto, tenor, bass)
		# The values represent MIDI note numbers

		ticks_per_beat = mid.ticks_per_beat
		# Let's use 16th notes (1/4 of a beat)
		ticks_per_step = ticks_per_beat // 4

		velocity = 100
		for voice in range(4):
			track = MidiTrack()
			mid.tracks.append(track)
			track.append(Message('program_change', program=voice, time=0))  # strings

			prev_note = 0
			now = 0
			last_event = 0
			for step, notes in enumerate(chorale):
					note_id = notes[voice]
					# Convert to integer MIDI note number
					midi_note = int(note_id)
					if midi_note == prev_note:
							pass
					else:
							track.append(Message('note_off', note=prev_note, velocity=0,
															time=now-last_event, channel=voice % 16))
							if (midi_note > 0) :
								track.append(Message('note_on', note=midi_note, velocity=velocity,
															time=0, channel=voice % 16))
							last_event = now
					prev_note = midi_note
					now = now + ticks_per_step
			track.append(Message('note_off', note=prev_note, velocity=0,
															time=now-last_event+ticks_per_step*12, channel=voice % 16))
# Save the MIDI file
		mid.save(midi_file)
		print(f"Saved MIDI file to {midi_file}")



In [19]:
extended = extend_sequence(X_test[5], 100)[window_size:]

array([[75., 67., 57., 50.],
       [75., 67., 57., 51.],
       [74., 67., 57., 51.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 67., 57., 52.],
       [74., 6

In [ ]:
quartet_to_midi(extended, data_root + "input.mid")

In [ ]:
!cp /usr/share/sounds/sf2/FluidR3_GM.sf2 ./font.sf2
!fluidsynth -ni font.sf2 /content/drive/MyDrive/Colab\ Notebooks/Bach\ Chorales/input.mid -F output.wav -r 44100
from IPython.display import Audio
Audio('output.wav')